# Modulo 8 · Lezione 3 — Backpropagation e Ottimizzazione

**Corso di Intelligenza Artificiale — Livello base**

---

## Cosa vediamo oggi?

| # | Argomento |
|---|------|
| 1 | La funzione di perdita: cosa minimizziamo? |
| 2 | Discesa del gradiente |
| 3 | Backpropagation: la regola della catena |
| 4 | Ottimizzatori: SGD, Momentum, Adam |
| 5 | Iperparametri dell'addestramento |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

np.random.seed(42)
print("Librerie pronte")

---
## 1. La funzione di perdita

```
La perdita (loss) misura QUANTO STA SBAGLIANDO il modello nelle predizioni.
L'obiettivo dell'addestramento: minimizzare la perdita.

REGRESSIONE:
  MSE (Mean Squared Error):
    L = (1/N) Σ (yᵢ - ŷᵢ)²
  MAE (Mean Absolute Error):
    L = (1/N) Σ |yᵢ - ŷᵢ|

CLASSIFICAZIONE BINARIA:
  Binary Cross-Entropy:
    L = -(1/N) Σ [yᵢ·log(ŷᵢ) + (1-yᵢ)·log(1-ŷᵢ)]
  Penalizza MOLTO le predizioni molto sicure e sbagliate.
  Se y=1 e ŷ≈0 → -log(0) → ∞

CLASSIFICAZIONE MULTICLASSE:
  Categorical Cross-Entropy:
    L = -(1/N) Σᵢ Σⱼ yᵢⱼ·log(ŷᵢⱼ)
  yᵢⱼ vale 1 solo per la classe corretta (one-hot).
  Equivale a: L = -(1/N) Σ log(ŷ[classe_corretta])
```

In [ ]:
# Visualizzare la binary cross-entropy
p = np.linspace(0.001, 0.999, 300)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

bce_y1 = -np.log(p)       # quando y=1
bce_y0 = -np.log(1 - p)   # quando y=0

axes[0].plot(p, bce_y1, 'steelblue', lw=2.5, label='y=1 (positivo reale)')
axes[0].plot(p, bce_y0, 'tomato',    lw=2.5, label='y=0 (negativo reale)')
axes[0].set_xlabel('Predizione ŷ (probabilità classe 1)')
axes[0].set_ylabel('Perdita')
axes[0].set_title('Binary Cross-Entropy')
axes[0].set_ylim(0, 5); axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Cross-entropy vs MSE
axes[1].plot(p, bce_y1, 'steelblue', lw=2.5, label='BCE (y=1)')
axes[1].plot(p, (1-p)**2, 'seagreen', lw=2.5, label='MSE (y=1)')
axes[1].set_xlabel('Predizione ŷ')
axes[1].set_title('BCE vs MSE: perché BCE?')
axes[1].set_ylim(0, 4); axes[1].legend(); axes[1].grid(True, alpha=0.3)
axes[1].text(0.05, 3.2, 'BCE penalizza più fortemente\nle predizioni sbagliate\n→ gradienti più grandi',
             fontsize=9, bbox=dict(boxstyle='round', facecolor='lightyellow'))

plt.tight_layout(); plt.show()

---
## 2. Discesa del gradiente

```
Intuizione: immagina di essere su una montagna nella nebbia e voler scendere.
  Non puoi vedere la valle, ma puoi sentire la pendenza sotto i piedi.
  → Fai passi nella direzione di maggiore discesa.

Matematicamente:
  Il gradiente ∇L punta verso la MASSIMA SALITA della perdita.
  Per minimizzare → muoversi nella direzione OPPOSTA.

  Aggiornamento dei pesi:
    w ← w - η · ∂L/∂w

  dove η (eta) = learning rate (tasso di apprendimento)

Varianti della discesa del gradiente:
  Batch GD:     usa TUTTI i dati per calcolare il gradiente → esatto ma lento
  Stochastic GD (SGD): usa UN campione → veloce ma molto rumoroso
  Mini-batch GD: usa un lotto piccolo (32-256) → equilibrio ← il più usato

Epoch = un passaggio completo su tutti i dati di addestramento.
```

In [ ]:
# Discesa del gradiente in 1D
def f(w):  return w**2 + 2*w + 1   # funzione di perdita semplice
def df(w): return 2*w + 2           # derivata (gradiente)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
w_range = np.linspace(-4, 2, 300)

for ax, lr, titulo in zip(axes, [0.05, 0.5, 1.5], ['lr=0.05 (lento)', 'lr=0.5 (ottimale)', 'lr=1.5 (oscilla)']):
    ax.plot(w_range, f(w_range), 'gray', lw=2, label='L(w)')
    ax.axhline(0, color='black', lw=0.5, ls='--')

    w = 3.0  # punto iniziale
    trayectoria_w = [w]
    trayectoria_l = [f(w)]
    for _ in range(20):
        w = w - lr * df(w)
        trayectoria_w.append(w)
        trayectoria_l.append(f(w))

    ax.plot(trayectoria_w, trayectoria_l, 'ro-', ms=5, lw=1.5, label=f'{len(trayectoria_w)} passi')
    ax.plot(trayectoria_w[0],  trayectoria_l[0],  'go', ms=10, label='Inizio')
    ax.plot(trayectoria_w[-1], trayectoria_l[-1], 'b*', ms=12, label='Finale')
    ax.set_xlabel('w'); ax.set_ylabel('L(w)')
    ax.set_title(titulo)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    ax.set_ylim(-0.5, 15)

plt.suptitle('Discesa del gradiente: effetto del learning rate', fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
# Discesa del gradiente in 2D (superficie di perdita)
def f2d(w1, w2):
    return (w1 - 1)**2 + 5*(w2 + 0.5)**2

w1_range = np.linspace(-3, 4, 100)
w2_range = np.linspace(-3, 2, 100)
W1, W2 = np.meshgrid(w1_range, w2_range)
Z = f2d(W1, W2)

fig, ax = plt.subplots(figsize=(8, 6))
levels = np.geomspace(0.1, 50, 20)
cs = ax.contourf(W1, W2, Z, levels=levels, cmap='RdYlGn_r', alpha=0.8)
ax.contour(W1, W2, Z, levels=levels, colors='white', alpha=0.3, linewidths=0.5)
plt.colorbar(cs, ax=ax, label='L(w₁,w₂)')

# Traiettoria del gradiente
lr = 0.15
w1, w2 = -2.5, 1.5  # inizio
tray = [(w1, w2)]
for _ in range(40):
    g1 = 2*(w1 - 1)
    g2 = 10*(w2 + 0.5)
    w1 -= lr * g1; w2 -= lr * g2
    tray.append((w1, w2))

tray = np.array(tray)
ax.plot(tray[:,0], tray[:,1], 'w-o', ms=4, lw=1.5, label='Traiettoria GD')
ax.plot(tray[0,0],  tray[0,1],  'go', ms=12, label='Inizio', zorder=10)
ax.plot(tray[-1,0], tray[-1,1], 'b*', ms=14, label='Finale', zorder=10)
ax.plot(1.0, -0.5, 'r*', ms=16, label='Minimo reale', zorder=10)

ax.set_xlabel('w₁'); ax.set_ylabel('w₂')
ax.set_title('Discesa del gradiente in 2D')
ax.legend(loc='upper right'); plt.tight_layout(); plt.show()

---
## 3. Backpropagation

```
Problema: una rete con milioni di pesi.
  Come calcoliamo ∂L/∂wᵢ per OGNI peso in modo efficiente?

Backpropagation = applicazione ricorsiva della regola della catena.

Regola della catena:
  Se L = f(g(x)), allora:
    dL/dx = (dL/df) · (df/dg) · (dg/dx)

In una rete:
  x → [z₁=W₁x+b₁] → [a₁=ReLU(z₁)] → [z₂=W₂a₁+b₂] → [ŷ=σ(z₂)] → L

  ∂L/∂W₁ = ∂L/∂ŷ · ∂ŷ/∂z₂ · ∂z₂/∂a₁ · ∂a₁/∂z₁ · ∂z₁/∂W₁

Algoritmo:
  1. FORWARD PASS: calcolare tutte le attivazioni a₁, a₂, ... e la perdita L
  2. BACKWARD PASS: calcolare i gradienti dall'uscita verso l'ingresso
     δ_uscita = ∂L/∂z_out (errore nell'ultimo strato)
     δ_strato = (Wᵀ · δ_successivo) ⊙ f'(z_strato)
     ∂L/∂W   = δ_strato · aᵀ_precedente
     ∂L/∂b   = δ_strato
  3. AGGIORNARE i pesi: w ← w - η · ∂L/∂w

Chiave dell'efficienza:
  Ogni gradiente si calcola UNA sola volta e si riutilizza.
  Complessità ≈ 2× forward pass.
```

In [ ]:
# Backpropagation da zero — rete 2→3→1 (classificazione binaria)
class MLPConBackprop:
    def __init__(self, n_entrada, n_oculta, n_salida=1, lr=0.1):
        self.lr = lr
        # Inizializzazione He
        self.W1 = np.random.randn(n_entrada, n_oculta) * np.sqrt(2/n_entrada)
        self.b1 = np.zeros(n_oculta)
        self.W2 = np.random.randn(n_oculta, n_salida) * np.sqrt(2/n_oculta)
        self.b2 = np.zeros(n_salida)

    def sigmoid(self, z): return 1 / (1 + np.exp(-np.clip(z, -500, 500)))
    def relu(self, z):    return np.maximum(0, z)
    def d_relu(self, z):  return (z > 0).astype(float)

    def forward(self, X):
        self.X  = X
        self.z1 = X @ self.W1 + self.b1
        self.a1 = self.relu(self.z1)
        self.z2 = self.a1 @ self.W2 + self.b2
        self.a2 = self.sigmoid(self.z2)
        return self.a2

    def backward(self, y):
        N = len(y)
        y = y.reshape(-1, 1)

        # Gradiente sull'uscita (BCE + sigmoid → forma semplificata)
        dL_dz2 = (self.a2 - y) / N                   # (N, 1)

        # Gradienti di W2 e b2
        self.dW2 = self.a1.T @ dL_dz2                 # (n_oculta, 1)
        self.db2 = dL_dz2.sum(axis=0)                 # (1,)

        # Propagare allo strato nascosto
        dL_da1  = dL_dz2 @ self.W2.T                  # (N, n_oculta)
        dL_dz1  = dL_da1 * self.d_relu(self.z1)       # (N, n_oculta)

        # Gradienti di W1 e b1
        self.dW1 = self.X.T @ dL_dz1                  # (n_entrada, n_oculta)
        self.db1 = dL_dz1.sum(axis=0)                 # (n_oculta,)

    def step(self):
        self.W1 -= self.lr * self.dW1
        self.b1 -= self.lr * self.db1
        self.W2 -= self.lr * self.dW2
        self.b2 -= self.lr * self.db2

    def loss(self, y_pred, y):
        y = y.reshape(-1, 1)
        eps = 1e-9
        return -np.mean(y*np.log(y_pred+eps) + (1-y)*np.log(1-y_pred+eps))

    def predict(self, X):
        return (self.forward(X) >= 0.5).astype(int).ravel()


# Addestrare sul dataset XOR (che il percettrone non poteva risolvere)
X_xor = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y_xor = np.array([0, 1, 1, 0], dtype=float)

red = MLPConBackprop(n_entrada=2, n_oculta=4, lr=0.5)
historia_loss = []

for epoch in range(5000):
    y_pred = red.forward(X_xor)
    l = red.loss(y_pred, y_xor)
    historia_loss.append(l)
    red.backward(y_xor)
    red.step()

print(f"Predizioni XOR: {red.predict(X_xor)}")
print(f"Etichette reali: {y_xor.astype(int)}")
print(f"Accuracy: {(red.predict(X_xor) == y_xor).mean():.2%}")
print(f"Loss finale: {historia_loss[-1]:.6f}")

plt.figure(figsize=(8, 3))
plt.plot(historia_loss, 'steelblue', lw=1.5)
plt.xlabel('Epoca'); plt.ylabel('BCE Loss')
plt.title('La backpropagation risolve XOR: curva di perdita')
plt.yscale('log'); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## 4. Ottimizzatori: SGD, Momentum, Adam

```
La discesa del gradiente di base ha dei problemi:
  - Lenta in valli strette (oscilla da una parete all'altra)
  - Sensibile al learning rate
  - Si blocca in minimi locali / punti di sella

MOMENTUM (Polyak, 1964):
  Accumula velocità nella direzione del gradiente.
  v ← β·v + (1-β)·∇L
  w ← w - η·v
  β tipico = 0.9 ("ricorda" il 90% della velocità precedente)

RMSPROP (Hinton, 2012):
  Adatta il lr per feature: divide il gradiente per la sua magnitudine recente.
  s ← β·s + (1-β)·∇L²
  w ← w - η·∇L / √(s + ε)

ADAM (Adaptive Moment Estimation, Kingma & Ba, 2014):
  Combina Momentum + RMSProp.
  m ← β₁·m + (1-β₁)·∇L      (momento del 1° ordine)
  v ← β₂·v + (1-β₂)·∇L²     (momento del 2° ordine)
  m̂ = m/(1-β₁ᵗ)              (correzione del bias)
  v̂ = v/(1-β₂ᵗ)
  w ← w - η·m̂/√(v̂+ε)
  β₁=0.9, β₂=0.999, ε=1e-8, η=0.001

Regola pratica: inizia con Adam. Se overfitting, prova SGD+Momentum.
```

In [ ]:
# Confrontare gli ottimizzatori su una funzione di esempio
def rosenbrock(w1, w2, a=1, b=100):
    return (a - w1)**2 + b*(w2 - w1**2)**2

def grad_rosenbrock(w1, w2, a=1, b=100):
    dw1 = -2*(a - w1) - 4*b*w1*(w2 - w1**2)
    dw2 = 2*b*(w2 - w1**2)
    return np.array([dw1, dw2])

class SGD:
    def __init__(self, lr): self.lr = lr; self.w = None
    def init(self, w0): self.w = w0.copy()
    def step(self, grad): self.w -= self.lr * grad; return self.w.copy()

class Momentum:
    def __init__(self, lr, beta=0.9):
        self.lr = lr; self.beta = beta; self.v = None
    def init(self, w0): self.w = w0.copy(); self.v = np.zeros_like(w0)
    def step(self, grad):
        self.v = self.beta*self.v + (1-self.beta)*grad
        self.w -= self.lr * self.v; return self.w.copy()

class Adam:
    def __init__(self, lr=0.01, b1=0.9, b2=0.999, eps=1e-8):
        self.lr=lr; self.b1=b1; self.b2=b2; self.eps=eps; self.t=0
    def init(self, w0):
        self.w=w0.copy(); self.m=np.zeros_like(w0); self.v=np.zeros_like(w0)
    def step(self, grad):
        self.t += 1
        self.m = self.b1*self.m + (1-self.b1)*grad
        self.v = self.b2*self.v + (1-self.b2)*grad**2
        m_hat = self.m / (1 - self.b1**self.t)
        v_hat = self.v / (1 - self.b2**self.t)
        self.w -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)
        return self.w.copy()

w0 = np.array([-1.5, 1.5])
n_steps = 500

opts = [
    (SGD(lr=0.001),      'SGD', 'coral'),
    (Momentum(lr=0.001), 'Momentum', 'steelblue'),
    (Adam(lr=0.05),      'Adam', 'seagreen'),
]

# Superficie
w1r = np.linspace(-2, 2, 200); w2r = np.linspace(-0.5, 2, 200)
W1, W2 = np.meshgrid(w1r, w2r)
Z = rosenbrock(W1, W2)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (opt, nombre, color) in zip(axes, opts):
    opt.init(w0)
    tray = [w0.copy()]
    for _ in range(n_steps):
        g = grad_rosenbrock(*opt.w)
        tray.append(opt.step(g))
    tray = np.array(tray)

    ax.contourf(W1, W2, np.log(Z+1), levels=20, cmap='YlOrRd', alpha=0.7)
    ax.plot(tray[:,0], tray[:,1], color=color, lw=1.5, alpha=0.8)
    ax.plot(tray[0,0],  tray[0,1],  'ko', ms=8, label='Inizio')
    ax.plot(tray[-1,0], tray[-1,1], color=color, marker='*', ms=12)
    ax.plot(1, 1, 'w*', ms=14, label='Minimo (1,1)')
    dist_final = rosenbrock(tray[-1,0], tray[-1,1])
    ax.set_title(f'{nombre}\nLoss finale={dist_final:.4f}')
    ax.set_xlabel('w₁'); ax.set_ylabel('w₂')
    ax.legend(fontsize=8)

plt.suptitle('Confronto tra ottimizzatori — Funzione di Rosenbrock', fontsize=10)
plt.tight_layout(); plt.show()

---
## 5. Iperparametri dell'addestramento

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

X, y = make_moons(n_samples=500, noise=0.2, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_tr_n = scaler.fit_transform(X_tr)
X_te_n = scaler.transform(X_te)

# Confrontare diversi learning rate
lrs = [0.001, 0.01, 0.1, 1.0]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for lr in lrs:
    np.random.seed(42)
    red = MLPConBackprop(n_entrada=2, n_oculta=8, lr=lr)
    hist = []
    for _ in range(500):
        pred = red.forward(X_tr_n)
        hist.append(red.loss(pred, y_tr))
        red.backward(y_tr); red.step()
    acc = accuracy_score(y_te, red.predict(X_te_n))
    axes[0].plot(hist, label=f'lr={lr} (acc={acc:.2%})', lw=1.5)

axes[0].set_xlabel('Epoca'); axes[0].set_ylabel('BCE Loss')
axes[0].set_title('Effetto del learning rate')
axes[0].set_ylim(0, 1.0); axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

# Confrontare le dimensioni dello strato nascosto
for n_oculta in [2, 4, 8, 32]:
    np.random.seed(42)
    red = MLPConBackprop(n_entrada=2, n_oculta=n_oculta, lr=0.1)
    hist = []
    for _ in range(500):
        pred = red.forward(X_tr_n)
        hist.append(red.loss(pred, y_tr))
        red.backward(y_tr); red.step()
    acc = accuracy_score(y_te, red.predict(X_te_n))
    axes[1].plot(hist, label=f'hidden={n_oculta} (acc={acc:.2%})', lw=1.5)

axes[1].set_xlabel('Epoca'); axes[1].set_ylabel('BCE Loss')
axes[1].set_title('Effetto della dimensione dello strato nascosto')
axes[1].set_ylim(0, 1.0); axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

---
## Pratica

### Esercizio 1 — Discesa del gradiente nella regressione lineare

**Consegna:** Implementa la discesa del gradiente per addestrare una regressione lineare `ŷ = w·x + b` con MSE:

```python
np.random.seed(42)
X = np.random.randn(100)
y = 3*X + 1.5 + np.random.randn(100)*0.5  # y = 3x + 1.5 + rumore
```

1. Inizializza `w=0, b=0`
2. Aggiorna con: `∂MSE/∂w = (2/N)·Σ(ŷ-y)·x`, `∂MSE/∂b = (2/N)·Σ(ŷ-y)`
3. Addestra per 1000 epoche con lr=0.1
4. Plotta la curva di perdita e la retta appresa
5. Converge al w=3, b=1.5 reale?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

```python
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
X = np.random.randn(100)
y = 3*X + 1.5 + np.random.randn(100)*0.5

w, b = 0.0, 0.0
lr = 0.1; N = len(X)
historia = []

for _ in range(1000):
    y_hat = w*X + b
    mse   = np.mean((y_hat - y)**2)
    historia.append(mse)
    dw = (2/N) * np.sum((y_hat - y) * X)
    db = (2/N) * np.sum(y_hat - y)
    w -= lr * dw; b -= lr * db

print(f"w appreso: {w:.4f}  (reale: 3.0)")
print(f"b appreso: {b:.4f}  (reale: 1.5)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(historia, 'steelblue', lw=2)
axes[0].set_xlabel('Epoca'); axes[0].set_ylabel('MSE')
axes[0].set_title('Curva di perdita'); axes[0].grid(True, alpha=0.3)

axes[1].scatter(X, y, s=20, alpha=0.7, label='Dati')
x_line = np.linspace(X.min(), X.max(), 100)
axes[1].plot(x_line, w*x_line + b, 'r-', lw=2, label=f'ŷ={w:.2f}x+{b:.2f}')
axes[1].set_title('Regressione lineare tramite GD'); axes[1].legend()
plt.tight_layout(); plt.show()
```
</details>

---
### Esercizio 2 — Regola della catena a mano

**Consegna:** Per la seguente rete a 1 neurone, calcola a mano e verifica con il codice:

```
Ingresso: x = 2.0
Pesi:     w = 0.5, b = -1.0
Etichetta: y = 1

Forward: z = w·x + b, a = sigmoid(z), L = BCE(y, a)
Backward: ∂L/∂w = ?, ∂L/∂b = ?

Nota: per sigmoid + BCE, la derivata semplificata è:
  ∂L/∂z = (a - y)
  ∂L/∂w = (a - y)·x
  ∂L/∂b = (a - y)
```

1. Calcola z, a, L a mano
2. Calcola ∂L/∂w e ∂L/∂b
3. Verifica numericamente: (L(w+ε) - L(w-ε)) / (2ε) ≈ ∂L/∂w?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 2</summary>

```python
import numpy as np

def sigmoid(z): return 1 / (1 + np.exp(-z))
def bce(y, a): return -(y*np.log(a+1e-9) + (1-y)*np.log(1-a+1e-9))

x = 2.0; w = 0.5; b = -1.0; y = 1

# Forward
z = w*x + b
a = sigmoid(z)
L = bce(y, a)
print(f"Forward:")
print(f"  z = {w}·{x} + {b} = {z:.4f}")
print(f"  a = σ({z:.4f}) = {a:.4f}")
print(f"  L = BCE({y}, {a:.4f}) = {L:.4f}")

# Backward analitico
dL_dz = a - y
dL_dw = dL_dz * x
dL_db = dL_dz
print(f"\nBackward (analitico):")
print(f"  ∂L/∂z = a - y = {dL_dz:.4f}")
print(f"  ∂L/∂w = {dL_dz:.4f} · {x} = {dL_dw:.4f}")
print(f"  ∂L/∂b = {dL_db:.4f}")

# Verifica numerica (differenze finite)
eps = 1e-5
grad_w_num = (bce(y, sigmoid((w+eps)*x+b)) - bce(y, sigmoid((w-eps)*x+b))) / (2*eps)
grad_b_num = (bce(y, sigmoid(w*x+(b+eps))) - bce(y, sigmoid(w*x+(b-eps)))) / (2*eps)
print(f"\nVerifica numerica (ε={eps}):")
print(f"  ∂L/∂w numerico: {grad_w_num:.4f}  (analitico: {dL_dw:.4f})")
print(f"  ∂L/∂b numerico: {grad_b_num:.4f}  (analitico: {dL_db:.4f})")
print(f"  Coincidono? {'SÌ' if abs(grad_w_num-dL_dw)<1e-4 else 'NO'}")
```
</details>

---
### Esercizio 3 — Implementare Adam da zero

**Consegna:** Aggiungi l'ottimizzatore Adam alla classe `MLPConBackprop`:

1. Modifica la classe affinché `__init__` accetti `optimizer='sgd'|'adam'`
2. Se Adam: mantieni `m`, `v`, `t` per W1, b1, W2, b2
3. Addestra su make_moons con SGD e Adam, lr=0.01, 300 epoche
4. Quale converge più velocemente?
5. Con quale lr Adam è ancora migliore di SGD con lr ottimizzato?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 3</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

X, y = make_moons(500, noise=0.2, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, 0.2, random_state=42)
sc = StandardScaler()
X_tr_n = sc.fit_transform(X_tr)
X_te_n = sc.transform(X_te)

class MLPAdam(MLPConBackprop):
    def __init__(self, n_entrada, n_oculta, lr=0.01, optimizer='sgd', b1=0.9, b2=0.999, eps=1e-8):
        super().__init__(n_entrada, n_oculta, lr=lr)
        self.optimizer = optimizer
        self.b1, self.b2, self.eps = b1, b2, eps
        self.t = 0
        self.m = {k: np.zeros_like(v) for k, v in
                  [('W1',self.W1),('b1',self.b1),('W2',self.W2),('b2',self.b2)]}
        self.v = {k: np.zeros_like(v) for k, v in
                  [('W1',self.W1),('b1',self.b1),('W2',self.W2),('b2',self.b2)]}

    def step(self):
        if self.optimizer == 'sgd':
            super().step(); return
        self.t += 1
        for nombre, param, grad in [
            ('W1', self.W1, self.dW1), ('b1', self.b1, self.db1),
            ('W2', self.W2, self.dW2), ('b2', self.b2, self.db2),
        ]:
            self.m[nombre] = self.b1*self.m[nombre] + (1-self.b1)*grad
            self.v[nombre] = self.b2*self.v[nombre] + (1-self.b2)*grad**2
            m_hat = self.m[nombre] / (1 - self.b1**self.t)
            v_hat = self.v[nombre] / (1 - self.b2**self.t)
            param -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

fig, ax = plt.subplots(figsize=(9, 4))
for opt_name, lr in [('sgd', 0.1), ('adam', 0.01)]:
    np.random.seed(42)
    red = MLPAdam(2, 8, lr=lr, optimizer=opt_name)
    hist = []
    for _ in range(300):
        pred = red.forward(X_tr_n)
        hist.append(red.loss(pred, y_tr))
        red.backward(y_tr); red.step()
    acc = accuracy_score(y_te, red.predict(X_te_n))
    ax.plot(hist, label=f'{opt_name.upper()} lr={lr} (acc={acc:.2%})', lw=2)

ax.set_xlabel('Epoca'); ax.set_ylabel('BCE Loss')
ax.set_title('SGD vs Adam'); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
```
</details>

---
### Esercizio 4 — Mini-batch vs Batch vs SGD

**Consegna:** Confronta le 3 modalità di discesa del gradiente su make_moons:

1. Batch GD: usa tutti i dati a ogni passo
2. SGD puro: usa 1 campione casuale a ogni passo
3. Mini-batch GD: lotti di 32 campioni

Per ciascuna: plotta la curva di perdita e misura l'accuracy finale.

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 4</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

def entrenar_modo(X_tr, y_tr, modo, n_epocas=300, lr=0.1, batch_size=32):
    np.random.seed(42)
    red = MLPConBackprop(2, 8, lr=lr)
    N = len(X_tr)
    hist = []
    for _ in range(n_epocas):
        if modo == 'batch':
            pred = red.forward(X_tr)
            hist.append(red.loss(pred, y_tr))
            red.backward(y_tr); red.step()
        elif modo == 'sgd':
            idx = np.random.randint(0, N)
            pred = red.forward(X_tr[[idx]])
            hist.append(red.loss(pred, y_tr[[idx]]))
            red.backward(y_tr[[idx]]); red.step()
        elif modo == 'mini-batch':
            idx = np.random.choice(N, batch_size, replace=False)
            pred = red.forward(X_tr[idx])
            hist.append(red.loss(pred, y_tr[idx]))
            red.backward(y_tr[idx]); red.step()
    return red, hist

fig, ax = plt.subplots(figsize=(9, 4))
colores_m = {'batch': 'steelblue', 'sgd': 'coral', 'mini-batch': 'seagreen'}
for modo in ['batch', 'sgd', 'mini-batch']:
    red, hist = entrenar_modo(X_tr_n, y_tr, modo)
    acc = accuracy_score(y_te, red.predict(X_te_n))
    # Smussare SGD per la visualizzazione
    if modo == 'sgd':
        suavizado = np.convolve(hist, np.ones(20)/20, mode='valid')
        ax.plot(suavizado, color=colores_m[modo], lw=2, alpha=0.5,
                label=f'{modo.upper()} (smussato, acc={acc:.2%})')
    else:
        ax.plot(hist, color=colores_m[modo], lw=2, label=f'{modo.upper()} (acc={acc:.2%})')

ax.set_xlabel('Epoca'); ax.set_ylabel('BCE Loss')
ax.set_title('Batch vs SGD vs Mini-batch')
ax.legend(); ax.grid(True, alpha=0.3); ax.set_ylim(0, 1.0)
plt.tight_layout(); plt.show()
```
</details>

---
### Esercizio 5 (sfida) — Verifica dei gradienti

**Consegna:** Implementa una funzione `verificar_gradientes(red, X, y, eps=1e-5)` che:

1. Calcola i gradienti analitici con `backward()`
2. Per ogni peso wᵢ, calcola il gradiente numerico: `(L(w+ε) - L(w-ε)) / (2ε)`
3. Confronta entrambi: errore relativo < 1e-5 → corretto
4. Restituisce l'errore massimo trovato

Questa tecnica si chiama "gradient check" e si usa per verificare le implementazioni.

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 5</summary>

```python
import numpy as np

def verificar_gradientes(red, X, y, eps=1e-5):
    # Calcolare i gradienti analitici
    pred = red.forward(X)
    red.backward(y)

    params  = [red.W1, red.b1, red.W2, red.b2]
    grads_a = [red.dW1, red.db1, red.dW2, red.db2]
    nombres = ['W1', 'b1', 'W2', 'b2']

    max_error = 0.0
    print(f"{'Parametro':8} {'Errore rel. max':>15}")
    print("-" * 28)

    for nombre, param, grad_a in zip(nombres, params, grads_a):
        grad_n = np.zeros_like(param)
        it = np.nditer(param, flags=['multi_index'], op_flags=['readwrite'])
        while not it.finished:
            idx = it.multi_index
            val_orig = param[idx]

            param[idx] = val_orig + eps
            L_mas = red.loss(red.forward(X), y)

            param[idx] = val_orig - eps
            L_men = red.loss(red.forward(X), y)

            param[idx] = val_orig
            grad_n[idx] = (L_mas - L_men) / (2 * eps)
            it.iternext()

        # Errore relativo
        dif = np.abs(grad_a - grad_n)
        denom = np.abs(grad_a) + np.abs(grad_n) + 1e-9
        err = (dif / denom).max()
        max_error = max(max_error, err)
        ok = '✓' if err < 1e-4 else '✗'
        print(f"{nombre:8} {err:>15.2e}  {ok}")

    print(f"\nErrore massimo globale: {max_error:.2e}")
    print(f"Implementazione {'CORRETTA ✓' if max_error < 1e-4 else 'NON CORRETTA ✗'}")
    return max_error

# Verificare la nostra implementazione
np.random.seed(42)
X_check = np.random.randn(5, 2)
y_check  = np.array([0, 1, 1, 0, 1], dtype=float)
red_check = MLPConBackprop(2, 3, lr=0.1)

error = verificar_gradientes(red_check, X_check, y_check)
```
</details>

---
## Riepilogo

```python
# Discesa del gradiente
w = w - lr * dL_dw           # aggiornamento base

# Backpropagation: regola della catena all'indietro
dL_dz_out = a_out - y        # gradiente sull'uscita (sigmoid + BCE)
dW_out    = a_hidden.T @ dL_dz_out
dL_da_h   = dL_dz_out @ W_out.T
dL_dz_h   = dL_da_h * (z_hidden > 0)  # derivata ReLU
dW_hidden = X.T @ dL_dz_h

# Adam (consigliato per iniziare)
m = beta1*m + (1-beta1)*grad
v = beta2*v + (1-beta2)*grad**2
w -= lr * (m/(1-beta1**t)) / (sqrt(v/(1-beta2**t)) + eps)
# Valori tipici: lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8

# In pratica: usa TensorFlow/Keras e lascia che calcoli i gradienti
# model.compile(optimizer='adam', loss='binary_crossentropy')
# → Lezione 4
```

## Cosa viene nella prossima lezione?
Nella **Lezione 4** iniziamo con **TensorFlow e Keras**: il framework che fa tutto questo automaticamente. Non scriverai mai più la backprop a mano.